# Error Classification with OpenAI + Gemini (With and Without RAG)

This notebook runs category classification on C++ code snippets using both OpenAI and Gemini models, then compares performance **without RAG** vs **with RAG** (retrieved examples + clean references).

## 1) Install dependencies (if needed)

Uncomment and run if packages are missing.

In [86]:
! pip install -q pandas scikit-learn openai google-generativeai python-dotenv langchain-community langchain-openai sentence-transformers faiss-cpu


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [121]:
from pathlib import Path
import os
import json
import re
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score

ROOT = Path('/Users/fereshteh/Zebra_AI/HintGenerator')
ARTIFACT_DIR = ROOT / 'artifacts'
DATASET_PATH = ARTIFACT_DIR / 'cpp_error_dataset.csv'
CATEGORIES_PATH = ARTIFACT_DIR / 'categories.json'
CLEAN_REFS_PATH = ARTIFACT_DIR / 'clean_references.jsonl'
FAISS_INDEX_DIR = ROOT / 'faiss_zbot_index'

# Set these in your environment before running:
# export OPENAI_API_KEY='...'
# export GEMINI_API_KEY='...'   (or GOOGLE_API_KEY)
# export OPENAI_MODEL='gpt-5.2'
# export GEMINI_MODEL='gemini-2.5-pro'

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', os.getenv('GOOGLE_API_KEY', ''))

# Recover from old misconfiguration where API keys were saved in *_MODEL env vars
if not OPENAI_API_KEY and os.getenv('OPENAI_MODEL', '').startswith('sk-'):
    OPENAI_API_KEY = os.getenv('OPENAI_MODEL', '')
if not GEMINI_API_KEY and os.getenv('GEMINI_MODEL', '').startswith('AIza'):
    GEMINI_API_KEY = os.getenv('GEMINI_MODEL', '')

OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.2')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-2.5-pro')

# If model env vars contain keys, reset them to real model names
if OPENAI_MODEL.startswith('sk-'):
    OPENAI_MODEL = 'gpt-4.1'
if GEMINI_MODEL.startswith('AIza'):
    GEMINI_MODEL = 'gemini-2.5-pro'

MAX_CODE_CHARS = 5000
RANDOM_STATE = 42
TEST_SIZE = 0.30
MAX_EVAL_ROWS = 80  # reduce API cost; set None to evaluate all test rows

print('Dataset exists:', DATASET_PATH.exists())
print('Categories exists:', CATEGORIES_PATH.exists())
print('Clean refs exists:', CLEAN_REFS_PATH.exists())
print('FAISS index dir exists:', FAISS_INDEX_DIR.exists())
print('OpenAI model:', OPENAI_MODEL)
print('Gemini model:', GEMINI_MODEL)
print('OpenAI key loaded:', bool(OPENAI_API_KEY))
print('Gemini key loaded:', bool(GEMINI_API_KEY))

Dataset exists: True
Categories exists: True
Clean refs exists: True
FAISS index dir exists: True
OpenAI model: gpt-4.1
Gemini model: gemini-2.5-pro
OpenAI key loaded: True
Gemini key loaded: True


In [122]:
df = pd.read_csv(DATASET_PATH)

# Build evaluation frame directly from dataset CSV labels
eval_df = df[['filename', 'relative_path', 'code_content', 'label', 'error_description']].copy()
eval_df['code_content'] = eval_df['code_content'].fillna('').astype(str)
eval_df['label'] = eval_df['label'].fillna('no error').astype(str).str.strip().str.lower()

# Categories come from CSV labels (not hard-coded prompt text)
categories = sorted(eval_df['label'].dropna().unique().tolist())

print('Rows for evaluation:', len(eval_df))
print('Categories from CSV labels:', categories)
eval_df['label'].value_counts().head(10)

Rows for evaluation: 115
Categories from CSV labels: ['coding error', 'colorsensor error', 'dcmotor error', 'distance sensor error', 'gyro error', 'no error', 'screen error']


label
no error                 51
coding error             29
dcmotor error            21
distance sensor error     4
gyro error                4
screen error              4
colorsensor error         2
Name: count, dtype: int64

In [123]:
# Load FAISS retriever from the prebuilt RAG index (same pattern as Agentic_Tutoring_System)
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

def get_embeddings_for_rag(use_openai: bool = True):
    if use_openai and os.environ.get('OPENAI_API_KEY'):
        print('Using OpenAI embeddings for FAISS query encoding: text-embedding-3-small')
        return OpenAIEmbeddings(model='text-embedding-3-small')
    print('Using HuggingFace embeddings for FAISS query encoding: all-MiniLM-L6-v2')
    return HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        model_kwargs={'device': 'cpu'}
    )

def load_faiss_retriever(index_dir: Path, top_k: int = 4, use_openai: bool = True):
    if not index_dir.exists():
        print(f'⚠️ FAISS index directory not found: {index_dir}')
        return None
    if not (index_dir / 'index.faiss').exists() or not (index_dir / 'index.pkl').exists():
        print(f'⚠️ Missing FAISS files in {index_dir}. Need index.faiss and index.pkl')
        return None

    embeddings = get_embeddings_for_rag(use_openai=use_openai)
    vectorstore = FAISS.load_local(
        str(index_dir),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(f'✅ Loaded FAISS index from {index_dir} ({vectorstore.index.ntotal} vectors)')
    return vectorstore.as_retriever(search_kwargs={'k': top_k})

retriever = load_faiss_retriever(FAISS_INDEX_DIR, top_k=4, use_openai=True)

Using OpenAI embeddings for FAISS query encoding: text-embedding-3-small
✅ Loaded FAISS index from /Users/fereshteh/Zebra_AI/HintGenerator/faiss_zbot_index (276 vectors)


In [101]:
def build_rag_context(query: str, retriever, top_k: int = 6):
    """Retrieve relevant documents and format as a structured context block."""
    docs = retriever.invoke(query)

    context_parts = []
    for i, doc in enumerate(docs[:top_k], 1):
        doc_type = str((doc.metadata or {}).get('type', 'unknown'))
        source = Path(str((doc.metadata or {}).get('source', ''))).name or 'unknown'
        snippet = str(doc.page_content).strip()[:600]

        if not snippet:
            continue

        label = {
            'curriculum': '📘 Curriculum',
            'library_reference': '📗 Library Docs',
            'mistake_pattern': '⚠️ Common Mistake',
            'cpp_example': '💻 Code Example',
        }.get(doc_type, '📄 Reference')

        context_parts.append(
            f"[{i}] {label} ({source})\n{snippet}\n{'─'*50}"
        )

    context_str = '\n\n'.join(context_parts).strip()
    return context_str, docs

def retrieve_context(code_text: str, top_k: int = 6) -> str:
    global retriever
    if retriever is None:
        return 'No RAG context available (FAISS retriever not loaded).'

    query = str(code_text or '').strip()[:2500]
    if not query:
        return 'No RAG context available (empty input code).'

    try:
        context_str, _ = build_rag_context(query, retriever, top_k=top_k)
        return context_str if context_str.strip() else 'No relevant context retrieved from FAISS index.'
    except AssertionError:
        print('⚠️ FAISS/query embedding dimension mismatch. Falling back to HuggingFace query embeddings.')
        retriever = load_faiss_retriever(FAISS_INDEX_DIR, top_k=top_k, use_openai=False)
        if retriever is None:
            return 'No RAG context available (failed to rebuild retriever after dimension mismatch).'
        try:
            context_str, _ = build_rag_context(query, retriever, top_k=top_k)
            return context_str if context_str.strip() else 'No relevant context retrieved from FAISS index.'
        except Exception:
            return 'No relevant context retrieved from FAISS index (retriever fallback failed).'
    except Exception:
        return 'No relevant context retrieved from FAISS index (retrieval error).'

def normalize_label(pred: str, valid_categories):
    p = str(pred).strip().lower()
    p = re.sub(r'[^a-z0-9 _-]', '', p)
    if p in valid_categories:
        return p
    for c in valid_categories:
        if c in p:
            return c
    return 'unknown'

def build_prompt(code_text: str, categories: list[str], use_rag: bool = False, context: str = ''):
    categories_block = '\n'.join([f'- {c}' for c in categories])

    core_prompt = f"""== TASK ==
You are an evaluator for robotics C++ error classification accuracy.
Choose exactly ONE label from the dataset label set.

== LABEL SET (from dataset CSV) ==
{categories_block}

== STUDENT'S C++ CODE ==
```cpp
{code_text[:MAX_CODE_CHARS].strip()}
```

Classification rules:
1) Predict only one label from LABEL SET.
2) Prioritize the dominant error category reflected in the code.
3) If no clear mistake exists, choose 'no error' only if it is in LABEL SET.
4) Do not invent labels and do not explain.

Return format:
Output ONLY the label text (exactly as one label from LABEL SET)."""

    if use_rag:
        return f"""{core_prompt}

== RETRIEVED CONTEXT (supporting references; may be noisy) ==
{context}

Use retrieved context only when relevant to improve label choice accuracy.
Still output only one label from LABEL SET."""

    return core_prompt

In [124]:
# API clients + model candidates for exploration
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', os.getenv('GOOGLE_API_KEY', ''))

OPENAI_MODEL_CANDIDATES = ['gpt-4o', 'gpt-4.1', 'gpt-5.2']
GEMINI_MODEL_CANDIDATES = ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-3-pro-preview']

openai_client = None
genai = None
gemini_model_cache = {}

def normalize_gemini_model_name(name: str) -> str:
    model = str(name or '').strip()
    if not model:
        return 'models/gemini-2.5-flash'
    if model.startswith('models/'):
        return model
    return f'models/{model}'

def get_gemini_model(model_name: str):
    normalized = normalize_gemini_model_name(model_name)
    if normalized in gemini_model_cache:
        return gemini_model_cache[normalized]
    if genai is None:
        return None
    try:
        gemini_model_cache[normalized] = genai.GenerativeModel(normalized)
    except Exception:
        gemini_model_cache[normalized] = None
    return gemini_model_cache[normalized]

if OPENAI_API_KEY:
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_API_KEY)

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)

print('OpenAI ready:', openai_client is not None)
print('Gemini ready:', genai is not None)
print('OpenAI candidates:', OPENAI_MODEL_CANDIDATES)
print('Gemini candidates:', GEMINI_MODEL_CANDIDATES)

OpenAI ready: True
Gemini ready: True
OpenAI candidates: ['gpt-4o', 'gpt-4.1', 'gpt-5.2']
Gemini candidates: ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-3-pro-preview']


In [125]:
valid_categories = [c.lower() for c in categories]

def classify_openai(code_text: str, use_rag: bool = False, model_name: str | None = None) -> tuple[str, float]:
    if openai_client is None:
        return 'unknown', float('nan')
    context = retrieve_context(code_text) if use_rag else ''
    prompt = build_prompt(code_text, valid_categories, use_rag=use_rag, context=context)
    chosen_model = model_name or OPENAI_MODEL

    start = time.perf_counter()
    try:
        resp = openai_client.responses.create(
            model=chosen_model,
            input=prompt,
            temperature=0
        )
        txt = getattr(resp, 'output_text', '') or ''
        label = normalize_label(txt, valid_categories)
    except Exception:
        label = 'unknown'
    latency_ms = (time.perf_counter() - start) * 1000
    return label, latency_ms

def classify_gemini(code_text: str, use_rag: bool = False, model_name: str | None = None) -> tuple[str, float]:
    chosen_model = model_name or GEMINI_MODEL
    model_obj = get_gemini_model(chosen_model)
    if model_obj is None:
        return 'unknown', float('nan')

    context = retrieve_context(code_text) if use_rag else ''
    prompt = build_prompt(code_text, valid_categories, use_rag=use_rag, context=context)

    start = time.perf_counter()
    try:
        resp = model_obj.generate_content(prompt, generation_config={'temperature': 0})
        txt = getattr(resp, 'text', '') or ''
        label = normalize_label(txt, valid_categories)
    except Exception:
        label = 'unknown'
    latency_ms = (time.perf_counter() - start) * 1000
    return label, latency_ms

In [114]:
# Build test set
train_df, test_df = train_test_split(
    eval_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=eval_df['label'] if eval_df['label'].nunique() > 1 else None
)

if MAX_EVAL_ROWS is not None:
    test_df = test_df.sample(min(MAX_EVAL_ROWS, len(test_df)), random_state=RANDOM_STATE).copy()

print('Train rows:', len(train_df))
print('Test rows used for API eval:', len(test_df))

Train rows: 80
Test rows used for API eval: 35


In [131]:
# OpenAI benchmark across requested models
benchmark_rows = []

for _, row in test_df.iterrows():
    code_text = str(row['code_content'])
    true_label = str(row['label']).lower()

    for model_name in OPENAI_MODEL_CANDIDATES:
        pred_no_rag, lat_no_rag = classify_openai(code_text, use_rag=False, model_name=model_name)
        pred_with_rag, lat_with_rag = classify_openai(code_text, use_rag=True, model_name=model_name)

        benchmark_rows.append({
            'filename': row['filename'],
            'true_label': true_label,
            'provider': 'openai',
            'model': model_name,
            'mode': 'no_rag',
            'pred_label': pred_no_rag,
            'latency_ms': lat_no_rag,
        })
        benchmark_rows.append({
            'filename': row['filename'],
            'true_label': true_label,
            'provider': 'openai',
            'model': model_name,
            'mode': 'with_rag',
            'pred_label': pred_with_rag,
            'latency_ms': lat_with_rag,
        })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.head()

,filename,true_label,provider,model,mode,pred_label,latency_ms
0,WrongSpellingofLibrary.cpp,no error,openai,gpt-4o,no_rag,dcmotor error,1437.640000
1,WrongSpellingofLibrary.cpp,no error,openai,gpt-4o,with_rag,dcmotor error,1517.053167
2,WrongSpellingofLibrary.cpp,no error,openai,gpt-4.1,no_rag,dcmotor error,530.598375
3,WrongSpellingofLibrary.cpp,no error,openai,gpt-4.1,with_rag,no error,680.343042
4,WrongSpellingofLibrary.cpp,no error,openai,gpt-5.2,no_rag,dcmotor error,874.201625


In [128]:
# Gemini benchmark across requested models
if 'benchmark_rows' not in globals():
    benchmark_rows = []

for _, row in test_df.iterrows():
    code_text = str(row['code_content'])
    true_label = str(row['label']).lower()

    for model_name in GEMINI_MODEL_CANDIDATES:
        pred_no_rag, lat_no_rag = classify_gemini(code_text, use_rag=False, model_name=model_name)
        pred_with_rag, lat_with_rag = classify_gemini(code_text, use_rag=True, model_name=model_name)

        benchmark_rows.append({
            'filename': row['filename'],
            'true_label': true_label,
            'provider': 'gemini',
            'model': model_name,
            'mode': 'no_rag',
            'pred_label': pred_no_rag,
            'latency_ms': lat_no_rag,
        })
        benchmark_rows.append({
            'filename': row['filename'],
            'true_label': true_label,
            'provider': 'gemini',
            'model': model_name,
            'mode': 'with_rag',
            'pred_label': pred_with_rag,
            'latency_ms': lat_with_rag,
        })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.tail()

,filename,true_label,provider,model,mode,pred_label,latency_ms
415,Inconsistance delay_solution.cpp,no error,gemini,gemini-2.5-flash,with_rag,coding error,6322.016875
416,Inconsistance delay_solution.cpp,no error,gemini,gemini-2.5-pro,no_rag,dcmotor error,15563.527708
417,Inconsistance delay_solution.cpp,no error,gemini,gemini-2.5-pro,with_rag,dcmotor error,12916.751459
418,Inconsistance delay_solution.cpp,no error,gemini,gemini-3-pro-preview,no_rag,unknown,158.010208
419,Inconsistance delay_solution.cpp,no error,gemini,gemini-3-pro-preview,with_rag,unknown,113.458458


In [132]:
def metric_block(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }

summary_rows = []
for (provider, model, mode), group in benchmark_df.groupby(['provider', 'model', 'mode']):
    metrics = metric_block(group['true_label'], group['pred_label'])
    summary_rows.append({
        'provider': provider,
        'model': model,
        'mode': mode,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'macro_precision': metrics['macro_precision'],
        'macro_recall': metrics['macro_recall'],
        'avg_latency_ms': group['latency_ms'].mean(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(['provider', 'model', 'mode'])
summary_df = summary_df.set_index(['provider', 'model', 'mode'])

latency_pivot = benchmark_df.pivot_table(
    index=['provider', 'model'],
    columns='mode',
    values='latency_ms',
    aggfunc='mean'
).rename(columns={
    'no_rag': 'no_rag_avg_latency_ms',
    'with_rag': 'with_rag_avg_latency_ms'
})

for col in ['no_rag_avg_latency_ms', 'with_rag_avg_latency_ms']:
    if col not in latency_pivot.columns:
        latency_pivot[col] = float('nan')

model_latency_df = latency_pivot[['no_rag_avg_latency_ms', 'with_rag_avg_latency_ms']].copy()
model_latency_df['avg_latency_ms'] = model_latency_df[['no_rag_avg_latency_ms', 'with_rag_avg_latency_ms']].mean(axis=1)
model_latency_df = model_latency_df[['avg_latency_ms', 'no_rag_avg_latency_ms', 'with_rag_avg_latency_ms']]

print('Latency summary by model:')
model_latency_df
summary_df

Latency summary by model:


accuracy  macro_f1  macro_precision  macro_recall  \
provider model   mode                                                          
openai   gpt-4.1 no_rag    0.771429  0.692481         0.669388      0.764881   
                 with_rag  0.800000  0.831652         0.826190      0.892857   
         gpt-4o  no_rag    0.600000  0.580519         0.515873      0.845238   
                 with_rag  0.600000  0.612605         0.588745      0.860119   
         gpt-5.2 no_rag    0.657143  0.618129         0.577359      0.729167   
                 with_rag  0.571429  0.547155         0.503663      0.687500   

                           avg_latency_ms  
provider model   mode                      
openai   gpt-4.1 no_rag        745.275489  
                 with_rag      858.199799  
         gpt-4o  no_rag        711.324572  
                 with_rag      766.771919  
         gpt-5.2 no_rag       1028.538577  
                 with_rag     1247.304175

In [126]:
# Quick smoke test across all requested models on one sample
sample_code = str(eval_df.iloc[0]['code_content'])

smoke_rows = []
for model_name in OPENAI_MODEL_CANDIDATES:
    pred, lat = classify_openai(sample_code, use_rag=False, model_name=model_name)
    smoke_rows.append({'provider': 'openai', 'model': model_name, 'pred': pred, 'latency_ms': round(lat, 2)})

for model_name in GEMINI_MODEL_CANDIDATES:
    pred, lat = classify_gemini(sample_code, use_rag=False, model_name=model_name)
    smoke_rows.append({'provider': 'gemini', 'model': model_name, 'pred': pred, 'latency_ms': round(lat, 2)})

smoke_df = pd.DataFrame(smoke_rows)
smoke_df

,provider,model,pred,latency_ms
0,openai,gpt-4o,coding error,1112.31
1,openai,gpt-4.1,coding error,714.33
2,openai,gpt-5.2,coding error,2432.04
3,gemini,gemini-2.5-flash,coding error,2291.25
4,gemini,gemini-2.5-pro,coding error,6512.49
5,gemini,gemini-3-pro-preview,coding error,126008.69


In [ ]:
out_dir = ROOT / 'artifacts'
out_dir.mkdir(parents=True, exist_ok=True)

results_path = out_dir / 'llm_classification_results_all_models.csv'
summary_path = out_dir / 'llm_classification_summary_all_models.csv'
latency_path = out_dir / 'llm_latency_summary_by_model.csv'
report_path = out_dir / 'llm_classification_report_all_models.txt'

benchmark_df.to_csv(results_path, index=False)
summary_df.reset_index().to_csv(summary_path, index=False)
model_latency_df.reset_index().to_csv(latency_path, index=False)

best_row = summary_df.sort_values(['macro_f1', 'accuracy'], ascending=False).head(1).reset_index().iloc[0]
report_txt = (
    f"Best configuration by macro_f1:\n"
    f"provider={best_row['provider']}\n"
    f"model={best_row['model']}\n"
    f"mode={best_row['mode']}\n"
    f"accuracy={best_row['accuracy']:.4f}\n"
    f"macro_f1={best_row['macro_f1']:.4f}\n"
    f"macro_precision={best_row['macro_precision']:.4f}\n"
    f"macro_recall={best_row['macro_recall']:.4f}\n"
    f"avg_latency_ms={best_row['avg_latency_ms']:.2f}\n"
)
report_path.write_text(report_txt, encoding='utf-8')

print('Saved:', results_path)
print('Saved:', summary_path)
print('Saved:', latency_path)
print('Saved:', report_path)

Saved: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/llm_classification_results.csv
Saved: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/llm_classification_summary.csv
Saved: /Users/fereshteh/Zebra_AI/HintGenerator/artifacts/openai_with_rag_report.txt


## 2) Interpreting Comparison

- Compare `accuracy` and `macro_f1` in `summary_df`.
- `*_with_rag` should improve when retrieved examples are relevant.
- Review `artifacts/llm_classification_results.csv` for per-file differences.

If one model returns `unknown`, verify API key/model setup and rerun.